# Copy Files from ABFS Path to Mounted Lakehouse

This notebook shows how to copy a few files from an ABFS (Azure Blob File System) path into the **already-mounted Lakehouse** attached to this notebook, using `notebookutils`.

The default mount point for the attached Lakehouse is `/lakehouse/default`.

**Steps:**
1. Configure the ABFS source path and the files to copy.
2. Copy individual files from the ABFS source into the mounted Lakehouse `Files` area.
3. Verify the copied files.

## 1. Configure source and destination

Set the ABFS source folder. **All files** in that folder will be copied into the destination sub-folder of the mounted Lakehouse.

In [ ]:
import notebookutils

# ABFS source folder (all files in this folder will be copied)
# Format: abfss://<workspace>@onelake.dfs.fabric.microsoft.com/<lakehouse>.Lakehouse/Files/<folder>
source_abfs_path = "abfss://<workspace>@onelake.dfs.fabric.microsoft.com/<lakehouse>.Lakehouse/Files/source"

# Destination sub-folder inside the mounted Lakehouse Files area
destination_subfolder = "imported"

# Optional: only copy files matching these extensions. Set to None to copy every file.
file_extensions = None  # e.g. [".csv", ".json", ".parquet"]

## 2. Use the default mount point

Since the Lakehouse is already mounted, point at its default local path `/lakehouse/default`. The `Files` area lives under `/lakehouse/default/Files`.

In [ ]:
# Default mount point of the attached Lakehouse
local_mount_path = "/lakehouse/default"
print(f"Using mounted Lakehouse at: {local_mount_path}")

## 3. Copy the files (recursively)

The files live in **sub-folders** of the source path. Walk the source tree recursively with `notebookutils.fs.ls`, then copy each file into the mounted Lakehouse `Files` area, **preserving the relative sub-folder structure**. The `file:` scheme points at the local mounted path.

In [ ]:
import os

# Local destination folder inside the mounted Lakehouse Files area
local_dest_folder = os.path.join(local_mount_path, "Files", destination_subfolder)
os.makedirs(local_dest_folder, exist_ok=True)

allowed = tuple(ext.lower() for ext in file_extensions) if file_extensions else None


def list_files_recursive(path):
    """Yield (file_info, relative_path) for every file under `path`, recursing into sub-folders."""
    for e in notebookutils.fs.ls(path):
        if e.isDir:
            yield from list_files_recursive(e.path)
        else:
            # relative path of the file with respect to the source root
            rel = e.path[len(source_abfs_path):].lstrip("/")
            yield e, rel


print(f"Scanning {source_abfs_path} recursively...")

copied = 0
for e, rel in list_files_recursive(source_abfs_path):
    if allowed and not e.name.lower().endswith(allowed):
        continue

    dest_path = os.path.join(local_dest_folder, *rel.split("/"))
    os.makedirs(os.path.dirname(dest_path), exist_ok=True)

    dst = f"file:{dest_path}"
    print(f"Copying {rel} -> {dst}")
    notebookutils.fs.cp(e.path, dst)
    copied += 1

print(f"Done. Copied {copied} file(s).")

## 4. Verify the copied files

List the destination folder to confirm the files were copied successfully.

In [ ]:
for root, _, names in os.walk(local_dest_folder):
    for name in names:
        full_path = os.path.join(root, name)
        rel = os.path.relpath(full_path, local_dest_folder)
        size_kb = os.path.getsize(full_path) / 1024
        print(f"{rel} ({size_kb:.1f} KB)")